# RAMEC — Revisión Automática Multimodal de Entregables de Construcción

Notebook de **ejecución en Google Colab**: clona el repositorio, instala las dependencias,
descarga los pesos del modelo y ejecuta las validaciones.

**Antes de empezar:**

(Opcional) Activa GPU en *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
   La inferencia también funciona en CPU.

## 1. Configuración

In [ ]:
REPO = "BRIDERI/Ramec"
TAG  = "v1.0"              # etiqueta del Release que contiene los pesos best.pt

## 2. (Opcional) Verificar GPU

In [ ]:
!nvidia-smi || echo "Sin GPU: se usará CPU (la inferencia funciona igual)."

## 3. Clonar el repositorio

In [ ]:
!git clone https://github.com/{REPO}.git ramec
%cd ramec

## 4. Dependencias del sistema (Tesseract + Poppler)
No se instalan con pip; el idioma `spa` es necesario para leer texto en español.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils

## 5. Dependencias de Python

In [ ]:
!pip -q install -r requirements.txt

## 6. Descargar los pesos del modelo (desde el Release)
Deja `models/planos/best.pt` y `models/documentos/best.pt`.

In [ ]:
!REPO={REPO} TAG={TAG} bash scripts/download_models.sh

## 7. Subir los PDFs a revisar
Sube uno o más PDFs (planos y/o documentos). Se guardan en la carpeta `pdfs/`.

In [ ]:
import os
os.makedirs("pdfs", exist_ok=True)
from google.colab import files
print("Selecciona uno o más PDFs...")
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, os.path.join("pdfs", nombre))
!ls -lh pdfs

*Alternativa:* en vez de subir archivos, puedes montar tu Google Drive y apuntar a una
carpeta con PDFs (descomenta y ajusta la ruta).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p pdfs && cp /content/drive/MyDrive/ruta/a/tus/pdfs/*.pdf pdfs/

## 8. Ejecutar la validación

In [ ]:
!python src/infer.py --pdfs pdfs --salida outputs/Reporte_validacion.xlsx

## 9. Ver y descargar el reporte
Se genera un Excel con seis hojas de verificación.

In [ ]:
import pandas as pd
xls = pd.ExcelFile("outputs/Reporte_validacion.xlsx")
print("Hojas:", xls.sheet_names)
display(pd.read_excel(xls, "VALIDACION_PROFESIONAL"))

from google.colab import files
files.download("outputs/Reporte_validacion.xlsx")

## (Opcional) Reentrenar los modelos
Requiere el **dataset anotado en CVAT** en `data/raw/` (no incluido en el repo por
confidencialidad/tamaño). Con tu dataset disponible, descomenta:

In [ ]:
# !python src/convert.py --planos data/raw/planos --documentos data/raw/documentos --val-frac 0.15
# !python src/train.py --task both     # genera models/<task>/best.pt